In [5]:
import requests
import json
import pprint
import pandas as pd
from datetime import datetime
import pprint
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/74.0.3729.169 YaBrowser/19.6.1.153 Yowser/2.5 Safari/537.36'}
df = pd.read_csv("top_ten_whales_both.csv")

In [6]:
#We start by extracting more data about the top trader
#The information we want is trade history and historical net loss/profits
wallet1 = df["wallet"][0]
def get_trade_history(wallet_address, limit=10):
    #first we make sure the address is correctly formatted
    wallet_address = wallet_address.strip()
    #and that it exists
    if not wallet_address or wallet_address == "0x...":
        print("!!NO WALLET FOUND!!")
        return

    url = "https://data-api.polymarket.com/activity"
    params = {
        'user': wallet_address,
        'limit': limit,
    }

    print(f"requesting data for {wallet_address[:10]}")

    try:
        response = requests.get(url, params=params, headers=headers)
        print(response.status_code)
        data = response.json()

        if not data:
            print("NOTHING FOUND. RETURN TO BASE")
            return None

        return data

    except Exception as e:
        print(f"something went wrong: {e}.")
        return None

whale_data_1 = get_trade_history(wallet1, limit=10)
pprint.pp(whale_data_1)

requesting data for 0xe4de861d
200
[{'proxyWallet': '0xe4de861d59e174d3d153a5968bc122c66f30c949',
  'timestamp': 1783196917,
  'conditionId': '0x518df6d694ad4658fe6c284850c029c6f00c9b3609cf05f7757d1fecba1641f8',
  'type': 'TRADE',
  'size': 3411.91,
  'usdcSize': 1228.2876,
  'transactionHash': '0x1f42508b648d4c49d52e0aaa225b7e3c738b6477dbaeead794c75789ef1dff21',
  'price': 0.36,
  'asset': '69816346859095495747189627053263753156196009953825285858477796399815602774300',
  'side': 'BUY',
  'outcomeIndex': 1,
  'title': 'US-Iran 60 day negotiation period extended?',
  'slug': 'us-iran-60-day-negotiation-period-extended-20260624044855448',
  'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/us-iran-60-day-negotiation-period-extended-byptptpt-20260624041226371-PQeKwYW3ZgF_.jpg',
  'eventSlug': 'us-iran-60-day-negotiation-period-extended-20260624044855448',
  'outcome': 'No',
  'name': 'On.The.Spectrum',
  'pseudonym': 'Firsthand-Offense',
  'bio': '',
  'profileImage': '',
  'p

In [7]:
#what info is there in the response
print(f"There are {len(whale_data_1)} trades in the response.")
print(f"These are the keys of each trade: {whale_data_1[0].keys()}")

There are 10 trades in the response.
These are the keys of each trade: dict_keys(['proxyWallet', 'timestamp', 'conditionId', 'type', 'size', 'usdcSize', 'transactionHash', 'price', 'asset', 'side', 'outcomeIndex', 'title', 'slug', 'icon', 'eventSlug', 'outcome', 'name', 'pseudonym', 'bio', 'profileImage', 'profileImageOptimized'])


In [8]:
#Let's keep building the function to give us a table of useful info
wallet_address = wallet1
def get_trade_history(wallet_address, limit=1000):
    #first we make sure the address is correctly formatted
    wallet_address = wallet_address.strip()
    #and that it exists
    if not wallet_address or wallet_address == "0x...":
        print("!!NO WALLET FOUND!!")
        return

    url = "https://data-api.polymarket.com/activity"
    params = {
        'user': wallet_address,
        'limit': limit,
    }

    print(f"requesting data for {wallet_address[:10]}")

    try:
        response = requests.get(url, params=params, headers=headers)
        print(response.status_code)
        data = response.json()

        if not data:
            print("NOTHING FOUND. RETURN TO BASE")
            return None

        #make a empty list to host the info
        clean_list = []
        for entry in data:
            raw_ts = entry.get('timestamp')
            #Format it correctly
            if isinstance(raw_ts, str):
                dt_obj = datetime.fromisoformat(raw_ts.replace('Z', '+00:00'))
            elif isinstance(raw_ts, (int, float)):
                if raw_ts > 10**11: raw_ts /= 1000
                dt_obj = datetime.fromtimestamp(raw_ts)
            else:
                continue

            dt_str = dt_obj.strftime('%Y-%m-%d %H:%M')
            
            #Get the numbers
            shares = float(entry.get('size', 0.0))
            price = float(entry.get('price', 0.0))
            
            #build our columns
            clean_list.append({
                'date': dt_str,
                'trade_type': entry.get('type', 'TRADE'),
                'side': entry.get('side', ''),
                'outcome': entry.get('outcome', ''),
                'amount': shares,
                'price': price,
                'total': price * shares,
                'title': entry.get('title', ''),
                'user': entry.get('proxyWallet', ''),
            })

        return clean_list

    except Exception as e:
        print(f"something went wrong: {e}.")
        return None

trade_list1 = get_trade_history(wallet1, limit=1000)
df_trade_list1 = pd.DataFrame(trade_list1)

requesting data for 0xe4de861d
200


In [9]:
pd.set_option('display.max_colwidth', None)
df_trade_list1.to_csv("top_trader_tyler.csv")

In [3]:
df_trade_list1 = pd.read_csv("top_trader_tyler.csv")

In [10]:
df_trade_list1

,date,trade_type,side,outcome,amount,price,total,title,user
0,2026-07-04 16:28,TRADE,BUY,No,3411.910000,0.360000,1228.287600,US-Iran 60 day negotiation period extended?,0xe4de861d59e174d3d153a5968bc122c66f30c949
1,2026-07-04 16:27,TRADE,SELL,No,5021.960000,0.950000,4770.862000,Strait of Hormuz traffic returns to normal by July 15?,0xe4de861d59e174d3d153a5968bc122c66f30c949
2,2026-07-04 16:19,TRADE,BUY,No,426.130000,0.639695,272.593200,Strait of Hormuz traffic returns to normal by August 31?,0xe4de861d59e174d3d153a5968bc122c66f30c949
3,2026-07-04 16:18,TRADE,BUY,No,5000.000000,0.485876,2429.378600,Strait of Hormuz traffic returns to normal by September 30?,0xe4de861d59e174d3d153a5968bc122c66f30c949
4,2026-07-04 16:17,TRADE,BUY,No,15.880000,0.630000,10.004400,Strait of Hormuz traffic returns to normal by August 31?,0xe4de861d59e174d3d153a5968bc122c66f30c949
5,2026-07-04 16:17,TRADE,BUY,No,1603.980000,0.630000,1010.507400,Strait of Hormuz traffic returns to normal by August 31?,0xe4de861d59e174d3d153a5968bc122c66f30c949
6,2026-07-04 15:30,TRADE,BUY,No,20047.970000,0.200000,4009.594000,Strait of Hormuz traffic returns to normal by December 31?,0xe4de861d59e174d3d153a5968bc122c66f30c949
7,2026-07-03 20:12,YIELD,,,0.445200,0.000000,0.000000,,0xe4de861d59e174d3d153a5968bc122c66f30c949
8,2026-07-03 08:13,TRADE,BUY,No,10000.000000,0.406393,4063.933800,US-Iran 60 day negotiation period extended?,0xe4de861d59e174d3d153a5968bc122c66f30c949
9,2026-07-03 07:46,TRADE,BUY,Yes,1408.990000,0.613253,864.067700,Will Gigi Hadid be one of Taylor Swift's bridesmaids?,0xe4de861d59e174d3d153a5968bc122c66f30c949


In [11]:
#Let's only keep the purchases
df_trade_list1 = df_trade_list1[df_trade_list1["side"] == "BUY"]

In [12]:
print(f"The top holder of Tyler market has spent {round(df_trade_list1["total"].sum())} dollars on polymarket bets.")

The top holder of Tyler market has spent 35516 dollars on polymarket bets.
